In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

# Random Forests and Hyperparameter Tuning
We'll check now how to use and tune RandomForest

## Build a Model Pipeline
Let's load the clean Airbnb dataset in again 
We created it in a previous notebook, it should exists in `/home/jovyan/work/outputs/airbnb/clean_data`

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

file_path = f"/home/jovyan/work/outputs/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)
train_df, test_df = airbnb_df.randomSplit([.8, .2], seed=42)

categorical_cols = <TODO>
index_output_cols = <TODO>

string_indexer = StringIndexer(inputCols=<TODO>, outputCols=<TODO>, handleInvalid="skip")

numeric_cols = <TODO>

assembler_inputs = <TODO>
vec_assembler = VectorAssembler(inputCols=<TODO>, outputCol=<TODO>)

rf = RandomForestRegressor(labelCol=<TODO>, maxBins=250)
stages = [<TODO>]
pipeline = Pipeline(stages=stages)

## ParamGrid

In [ ]:
print(rf.explainParams())

There are a lot of hyperparameters we could tune, and it would take a long time to manually configure.
We can define a grid of hyperparameters to test:
  - **`maxDepth`**: max depth of each decision tree between **`2 and 5`**)
  - **`numTrees`**: number of decision trees to train between **`5 and 10`**)
**`addGrid()`** requires the name of the parameter as first input (e.g. **`rf.maxDepth`**), and a list of the possible values (e.g. **`[2, 5]`**).

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder

param_grid = (ParamGridBuilder()
              .addGrid(<TODO>, <TODO>)
              .addGrid(<TODO>, <TODO>)
              .build())

We pass in the **`estimator`** (pipeline), **`evaluator`**, and **`estimatorParamMaps`** to <a href="https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.CrossValidator.html?highlight=crossvalidator#pyspark.ml.tuning.CrossValidator" target="_blank">CrossValidator</a> so that it knows:
- Which model to use
- How to evaluate the model
- What hyperparameters to set for the model
We can also set the number of folds we want to split our data into (3), as well as setting a seed so we all have the same split in the data.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator

evaluator = <TODO>

cv = CrossValidator(estimator=<TODO>,
                    evaluator=<TODO>,
                    estimatorParamMaps=<TODO>, 
                    numFolds=3, seed=42)

This param grid will cause to evaluate the model with the following hyperparameters combination:
* maxDepth: 2 numTrees: 5
* maxDepth: 2 numTrees: 10
* maxDepth: 5 numTrees: 5
* maxDepth: 5 numTrees: 10
So four combinations

In [ ]:
cv_model = cv.fit(<TODO>)

Since we have things like StringIndexer (an estimator) in the pipeline, it will be recalculated entirely if the pipeline is put in the cross validator. And that step doesn't matter for the cross validation step
* We can put the only piece that changes (Regressor itself) in the cross validation

In [ ]:
cv = CrossValidator(estimator=<TODO>,
                    evaluator=<TODO>,
                    estimatorParamMaps=<TODO>, 
                    numFolds=3, seed=42)


pipeline = Pipeline(stages=[<TODO>])

pipeline_model = pipeline.fit(<TODO>)

We can look at the model with the best hyperparameter configuration by checking it's metrics

In [ ]:
list(zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics))

In [ ]:
pred_df = pipeline_model.transform(<TODO>)

rmse = evaluator.evaluate(<TODO>)
r2 = evaluator.setMetricName("r2").evaluate(<TODO>)
print(f"RMSE is {rmse}")
print(f"R2 is {r2}")